# mu-logsigma-encoder-head — faded example 1: Build an encoder head module with nn.Linear + chunk

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`. The last cell reports your progress on the `VAE: mu+logsigma encoder head` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: mu+logsigma encoder head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mu-logsigma-encoder-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mu-logsigma-encoder-head"
DD_SUBTOPIC = "VAE: mu+logsigma encoder head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The standard VAE encoder head uses a single `nn.Linear(d_in, 2 * latent_dim)` layer followed by `chunk(2, dim=-1)` to split the output into `(mu, logsigma)`. The key is doubling the output features of the linear layer so both parameter vectors come from a single weight matrix.

## Faded exercise 1

Implement `EncoderHead(nn.Module)` with:
- `__init__(self, d_in, latent_dim)`: call `super().__init__()`, store `self.proj = nn.Linear(d_in, 2 * latent_dim)`.
- `forward(self, h)`: project `h` and split with `chunk(2, dim=-1)`, returning `(mu, logsigma)`.

Also implement `build_encoder_head(d_in, latent_dim)` that returns an instance.

Your task: **fill in the forward method body**: the projection call, the chunk split, and the return statement.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch
import torch.nn as nn

class EncoderHead(nn.Module):
    def __init__(self, d_in: int, latent_dim: int):
        super().__init__()
        self.proj = nn.Linear(d_in, 2 * latent_dim)

    def forward(self, h: torch.Tensor):
        params = self.proj(h)
        mu, logsigma = params.chunk(2, dim=-1)
        return mu, logsigma

def build_encoder_head(d_in: int, latent_dim: int):
    return EncoderHead(d_in, latent_dim)

def _test():
    import torch
    torch.manual_seed(0)
    head = build_encoder_head(32, 8)
    h = torch.randn(5, 32)
    mu, logsigma = head(h)
    assert mu.shape == (5, 8)
    assert logsigma.shape == (5, 8)
    assert not torch.allclose(mu, logsigma)


def _test():
    import torch
    torch.manual_seed(0)
    head = build_encoder_head(32, 8)
    h = torch.randn(5, 32)
    mu, logsigma = head(h)
    assert mu.shape == (5, 8), f"mu shape wrong: {mu.shape}"
    assert logsigma.shape == (5, 8), f"logsigma shape wrong: {logsigma.shape}"
    # mu and logsigma should be different halves of the projection
    params_ref = head.proj(h)
    mu_ref, ls_ref = params_ref.chunk(2, dim=-1)
    assert torch.allclose(mu, mu_ref)
    assert torch.allclose(logsigma, ls_ref)
    # Parameters count: 32*16 + 16
    n_params = sum(p.numel() for p in head.parameters())
    assert n_params == 32 * 16 + 16, f"wrong param count: {n_params}"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn as nn

class EncoderHead(nn.Module):
    def __init__(self, d_in: int, latent_dim: int):
        super().__init__()
        self.proj = nn.Linear(d_in, 2 * latent_dim)

    def forward(self, h: torch.Tensor):
        params = self.proj(h)
        mu, logsigma = params.chunk(2, dim=-1)
        return mu, logsigma

def build_encoder_head(d_in: int, latent_dim: int):
    return EncoderHead(d_in, latent_dim)

def _test():
    import torch
    torch.manual_seed(0)
    head = build_encoder_head(32, 8)
    h = torch.randn(5, 32)
    mu, logsigma = head(h)
    assert mu.shape == (5, 8)
    assert logsigma.shape == (5, 8)
    assert not torch.allclose(mu, logsigma)
```
</details>